# Ollama on Colab, exposed over a cloudflared quick tunnel

Runtime -> Change runtime type -> T4 GPU, before running any cell.

This notebook installs Ollama, pulls a model, serves it, and opens a
`cloudflared` quick tunnel so `vishwakarma` (running on your local machine)
can reach this Colab GPU as the `ollama-colab` fallback provider.

The session is ephemeral: Colab disconnects after ~12h or on idle, and a
fresh session gets a new tunnel URL. Copy the last cell's output into
`OLLAMA_COLAB_BASE_URL` in your local `.env` after every restart -- see
`models.yaml`'s `ollama-colab` block and the README's "Colab-hosted Ollama
fallback" section for why it's an env var and not a line in `models.yaml`.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &

If this pull fails, the model tag doesn't exist in Ollama's library --
swap `MODEL_NAME` below for a real tag (e.g. `qwen2.5-coder:7b`,
`deepseek-coder-v2:16b`, `llama3.1:8b`), and make the matching one-line edit
to the `ollama-colab` candidates in `models.yaml`.

In [ ]:
MODEL_NAME = "deepseek-v4-pro"

import time
time.sleep(3)
!ollama pull {MODEL_NAME}

In [ ]:
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared

In [ ]:
!nohup ./cloudflared tunnel --url http://localhost:11434 > tunnel.log 2>&1 &

Public URL -- copy this into `.env` as `OLLAMA_COLAB_BASE_URL=<url>/v1`:

In [ ]:
import time
time.sleep(5)
!grep -o 'https://[a-zA-Z0-9-]*\.trycloudflare\.com' tunnel.log | head -1